# ADAPT quickstart

Find prompts that maximally activate a Gemma Scope latent in Gemma 2 2B. Runs in about 1.5 minutes per latent on one A40-class GPU.

Gemma 2 is gated on Hugging Face: accept the licence at https://huggingface.co/google/gemma-2-2b and run `huggingface-cli login` first.

In [ ]:
import json
import torch
from sae_lens import HookedSAETransformer, SAE
from ADAPT import ADAPT, OptimizationConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedSAETransformer.from_pretrained("gemma-2-2b", device=device)

Pick a latent. `latents.json` holds the 486 latents from the paper, with their Neuronpedia maximum activations; any latent of any Gemma Scope SAE works.

In [ ]:
latents = json.load(open("latents.json"))
layer, feature_id = 17, latents["latents"]["17"][0]["feature_id"]
max_act = latents["latents"]["17"][0]["max_act"]

sae = SAE.from_pretrained("gemma-scope-2b-pt-res-canonical", f"layer_{layer}/width_16k/canonical", device=device)
sae = sae[0] if isinstance(sae, tuple) else sae  # older sae_lens versions return a tuple

Run ADAPT with the paper's settings (`OptimizationConfig()`); pass `verbose=True` to print the population every iteration.

In [ ]:
torch.manual_seed(0)
optimizer = ADAPT(model, sae)
result = optimizer.optimize(feature_id, OptimizationConfig(), verbose=False)
print(f"{result.runtime:.0f}s")

The final population, best first. The activation ratio compares against the strongest activation Neuronpedia found in its dataset.

In [ ]:
state = optimizer.state
for i in state.activations.argsort(descending=True).tolist():
    prompt = model.tokenizer.decode(state.prompts[i][state.masks[i].bool()])
    act = state.activations[i].item()
    print(f"{act:7.2f}  (ratio {act / max_act:.2f})  {prompt!r}")